In [1]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'
REPO_DIR = Path('/content/Yolo-ST-GCN')
BRANCH = 'refactor-1'

# Install lightweight dependencies needed for smoke checks.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'numpy', 'scipy', 'torch', '-q'],
    check=True,
)

# Clone or update exact branch used for current development.
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', os.getcwd())
print('Branch:', BRANCH)

In [2]:
# !git pull
# print(1)

In [2]:
!python scripts/build_gym99_from_gym288.py --experiment_config configs/experiments/gym99_build_from_gym288.json

In [3]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
from huggingface_hub import snapshot_download

download_dir = Path('/content/Gym288-skeleton')
if not (download_dir.exists() and any(download_dir.iterdir())):
    print('Downloading Gym288-skeleton...')
    download_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='Lozumi/Gym288-skeleton',
        repo_type='dataset',
        local_dir=str(download_dir),
        local_dir_use_symlinks=False,
    )
else:
    print('Gym288-skeleton already exists.')

pkl_candidates = sorted(download_dir.rglob('*.pkl'))
if not pkl_candidates:
    raise FileNotFoundError('No .pkl file found for Gym288 dataset.')

gym288_pkl = str(pkl_candidates[0])
print('Gym288 pickle:', gym288_pkl)

In [5]:
# !python scripts/train_gym99.py \
# --auto_build_from_gym288 \
# --gym288_dataset_path /content/Gym288-skeleton/gym_288_skeleton.pkl \
# --dataset_path /content/Gym99-from-Gym288/gym99_from_gym288.pkl \
# --out_dir outputs/refactor1_2ep/gym99_coco18_2s \
# --epochs 2 \
# --batch_size 256 \
# --lr 0.001 \
# --num_workers 0 \
# --joint_spec_name coco18 \
# --use_two_stream

In [ ]:
# !git pull

In [5]:
!python scripts/train_gym99.py \
--auto_build_from_gym288 \
--gym288_dataset_path /content/Gym288-skeleton/gym_288_skeleton.pkl \
--dataset_path /content/Gym99-from-Gym288/gym99_from_gym288.pkl \
--out_dir outputs/refactor1_2ep/gym99_coco18_2s \
--epochs 30 \
--batch_size 256 \
--lr 0.001 \
--num_workers 0 \
--joint_spec_name coco18 \
--save_every_epochs 10 \
--use_two_stream \
--loss_name focal \
--focal_alpha_mode sqrt_inverse
# --max_train_samples 8192 \
# --max_val_samples 2048 \
# --train_data_mode preload_vram